# Custom rate nodes and the jit cache

Most models are built from the rate nodes summer4 ships. A package author who
reuses the *same* named process across many models can add their own node: a
`RateOps` subclass plus an evaluator registered with
`register_rate_eval`. `ForceOfInfection` is one.

A compiled model's identity is a **digest of its rate expressions**, and
`CompiledModel` is designed to be passed to
`jax.jit` as a static argument. So a custom node has one obligation: encode
every field that changes its value, in `__rate_bytes__`. If it does not, two
models that differ only in that node's fields compare equal, hash equal, and
share one compiled program — and the second silently returns the first's
numbers.

This page checks three claims:

1. Two models differing only in a custom node's field values run as two
   different models **under `jax.jit`**, matching an unjitted Euler loop.
2. Building *n* such models produces *n* distinct jit cache entries, not one.
3. A node class that omits `__rate_bytes__` is **refused at registration**, with
   a `TypeError` naming the class and the method it must implement.


## A reusable process as a rate node

`Shedding` is a frequency-dependent hazard — shedding rate times the infectious
fraction of a group — written as a named node so `Shedding(...)` can be dropped
into any flow. The evaluator composes it from `Reduce`, and
`__rate_bytes__` encodes all three fields: the grouping property, the selector
that says who is infectious, and the shedding rate itself.

The two models below differ **only** in that shedding rate. Their digests,
hashes and equality must all separate them.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from dataclasses import dataclass
from functools import partial
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np

from summer4 import FlowModel, Property, PropertyMap, Reduce, TransitionFlow
from summer4.flows.rates import RateOps, as_rate, register_rate_eval
from summer4.selectors import Selector

state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)


@dataclass(frozen=True, slots=True)
class Shedding(RateOps):
    """Shedding rate x infectious fraction of the group."""

    shedding: RateOps
    infectious: Selector
    group_by: Property

    def __rate_bytes__(self) -> bytes:
        from summer4.flows.rates import _rate_bytes, _selector_bytes

        return (
            b"shedding"
            + self.group_by.name.encode()
            + _selector_bytes(self.infectious)
            + _rate_bytes(self.shedding)
        )


@register_rate_eval(Shedding)
def _eval_shedding(expr: Shedding, *, eval_child: Any, **_: Any) -> Any:
    infectious = eval_child(Reduce(sum_over=expr.group_by, where=expr.infectious))
    everyone = eval_child(Reduce(sum_over=expr.group_by))
    return eval_child(expr.shedding) * infectious / everyone


def build(shedding: float) -> Any:
    """An SIR model whose only varying input is the shedding rate."""
    model = FlowModel(pmap)
    model.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            Shedding(as_rate(shedding), infectious=state["I"], group_by=pop),
        )
    )
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
    return model.compile()


slow = build(0.2)
fast = build(0.8)

assert slow._digest != fast._digest
assert slow != fast
assert hash(slow) != hash(fast)
assert build(0.2) == slow  # same inputs, same model: cache hits are still wanted
print("shedding 0.2 and 0.8 compile to different models; 0.2 rebuilds to the same one")


## The same jitted function, two models

`trajectory` takes the model as `static_argnums=0` — the documented way to use
a `CompiledModel` under `jax.jit`. Calling it
with `slow` and then with `fast` is precisely the situation the digest has to
get right: if the two models compared equal, the second call would reuse the
first's compiled program.

The figure is prevalent infection `I(t)` for both. The two curves should be
clearly different — a faster shedding rate gives an earlier, taller peak — and
each jitted curve should sit on top of an unjitted Euler loop over the same
model.


In [ ]:
DT = 0.1
STEPS = 401
Y0 = np.array([990.0, 10.0, 0.0])


@partial(jax.jit, static_argnums=(0, 3))
def trajectory(model: Any, y0: Any, dt: float, steps: int) -> Any:
    """Euler-integrate ``model`` under jit, with the model static."""

    def step(y: Any, _: Any) -> Any:
        return y + dt * model.vector_field(0.0, y, {}), y

    _, ys = jax.lax.scan(step, y0, jnp.arange(steps))
    return ys


def eager_trajectory(model: Any) -> Any:
    """The same Euler loop in NumPy, with no tracing and no cache."""
    y = Y0.copy()
    out = []
    for _ in range(STEPS):
        out.append(y.copy())
        y = y + DT * np.asarray(model.vector_field(0.0, y, {}))
    return np.array(out)


jit_slow = np.asarray(trajectory(slow, jnp.asarray(Y0), DT, STEPS))
jit_fast = np.asarray(trajectory(fast, jnp.asarray(Y0), DT, STEPS))

np.testing.assert_allclose(jit_slow, eager_trajectory(slow), rtol=1e-4, atol=1e-3)
np.testing.assert_allclose(jit_fast, eager_trajectory(fast), rtol=1e-4, atol=1e-3)

peak_slow = jit_slow[:, 1].max()
peak_fast = jit_fast[:, 1].max()
assert peak_fast > 2.0 * peak_slow
assert jit_slow[:, 1].argmax() > jit_fast[:, 1].argmax()
print(f"peak I: shedding 0.2 -> {peak_slow:.1f}, shedding 0.8 -> {peak_fast:.1f}")

times = np.arange(STEPS) * DT
pd.DataFrame(
    {
        "shedding 0.2 (jit)": jit_slow[:, 1],
        "shedding 0.8 (jit)": jit_fast[:, 1],
        "shedding 0.2 (eager)": eager_trajectory(slow)[:, 1],
        "shedding 0.8 (eager)": eager_trajectory(fast)[:, 1],
    },
    index=times,
).plot(
    title="Two shedding rates through one jitted function",
    labels={"index": "time", "value": "infectious"},
)


## One cache entry per model, not one for all of them

The structural version of the same claim. Build a sweep of models that differ
only in the shedding rate and count the **distinct** digests as the sweep grows.
The count should track the number of models exactly — the diagonal. A node whose
digest ignored its fields would flatten this line to 1, and every model after the
first would return the first one's numerics.


In [ ]:
sweep_rates = [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
sweep = [build(rate) for rate in sweep_rates]

seen: set[bytes] = set()
distinct = []
for model in sweep:
    seen.add(model._digest)
    distinct.append(len(seen))

built = list(range(1, len(sweep) + 1))
assert distinct == built
assert len({hash(model) for model in sweep}) == len(sweep)
print(f"{distinct[-1]} distinct jit cache entries for {len(sweep)} models")

pd.DataFrame(
    {"distinct cache entries": distinct, "models built": built},
    index=sweep_rates,
).plot(
    title="Distinct jit cache entries as the shedding sweep grows",
    labels={"index": "shedding rate", "value": "count"},
)


## The guard: a node without `__rate_bytes__` is refused

The protection is not advice in a docstring. `register_rate_eval` inspects the
class and raises `TypeError` when it cannot digest it, so the mistake lands at
the class definition — at import — instead of as wrong numbers much later.
Nothing is added to the evaluator registry.


In [ ]:
@dataclass(frozen=True, slots=True)
class Unkeyed(RateOps):
    """Deliberately broken: no ``__rate_bytes__``."""

    shedding: RateOps


refused = None
try:
    register_rate_eval(Unkeyed)(lambda expr, **_: 0.0)
except TypeError as exc:
    refused = str(exc)

assert refused is not None, "registration should have raised"
assert "Unkeyed" in refused
assert "__rate_bytes__" in refused

from summer4.flows.rates import _RATE_EVALUATORS

assert Unkeyed not in _RATE_EVALUATORS
assert Shedding in _RATE_EVALUATORS
print(refused)


## What to remember

- A custom rate node **must** implement `__rate_bytes__`, encoding every field
  that changes its value. Registration enforces it.
- Follow the shape used above and by `summer4.epi.ForceOfInfection`: a tag for
  the class, then each field through a stable encoder (`_rate_bytes` for child
  rate expressions, `_selector_bytes` for selectors).
- Identical inputs should still compare equal — that is what makes the jit cache
  useful. The requirement is that *different* inputs compare different.
- `docs/cookbook/01-custom-rates.ipynb` walks the three rungs of custom rates
  for modellers; this page is the correctness gate for the top rung.
